# Configuración de Big Query (modelos de entidad y sus esquemas de datos)

## Creación del entorno virtual de Python

Antes de nada es preciso generar un entorno virtual con Python y activarlo con los siguientes comandos.:

```powershell 

python -m venv .venv

.\.venv\Scripts\activate

```

Para posteriormente hacer la carga de los requisitos (librerías y/o módulos):

```powershell

pip install -r .\requirements.txt

```

## Conexión a la infraestructura y definición del Dataset

Se realizaría como sigue a continuación:

In [7]:
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv
load_dotenv()

# Ruta absoluta al proyecto
carpeta = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Configuración
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.path.join(carpeta, "..credentials", "hglebredo-online-92ccd6898cd9.json")

# Cliente autenticado
CREDENTIALS_PATH = os.path.abspath(
    os.path.join(
        carpeta,
        os.pardir,
        "credentials",
        "hglebredo-online-92ccd6898cd9.json",
    )
)

if not os.path.isfile(CREDENTIALS_PATH):
    raise FileNotFoundError(f"Credentials file not found: {CREDENTIALS_PATH}")

credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

# Crear dataset
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"  # Datos en Europa
dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset {DATASET_ID} creado")

Dataset onlinemarket_dataset creado


## Creación de entidades y orden de carga.

El orden de carga ha de ser siempre `Padre → Hijo` y la realizaré según indica el siguiente esquema:

```text
customers   ─┐
categories  ─┤  (sin FK → se pueden cargar en paralelo)
             │
 products   ─┘  (FK → categories)
   │
 orders  (FK → customers)
   │
order_items (FK → orders + products)
   │
payments    (FK → orders)
   │
reviews     (FK → order_items)
```


### Mapeo de tipos a la creación

Resumen de decisiones de mapeo de tipos entre lenguaje `SQL` y `GibQuery`:

| SQL | BIGQUERY | MOTIVO |  
| --- | ------| -------|
| INT AUTO_INCREMENT | INT64 | No vi que BigQuery tuvierqa incremento automático. |  
| VARCHAR(n) / TEXT | STRING | BigQuery usa STRING sin longitud |  
| NUMERIC(p,s) | NUMERIC | BigQuery NUMERIC (decimal de precisión estándar) |  
| ENUM(...)  | STRING |z No he visto que tenga tipo ENUM |  
| BOOLEAN | BOOLEAN | Idéntico |  
| TIMESTAMP | TIMESTAMP | Directo |  
| CHECK(...) | — | Sin equivalente; validar en la app |  
| PRIMARY KEY (a, b) | Dos campos REQUIRED | En BigQuery **no se declaran PK compuestas***  |   

**(*) La integridad se maneja en ejecución.**

Si además se quieren documentar las FK de la tabla, parece que BigQuery permite definirlas como metadatos pero no las maneja como tales Foreign Keys (añadir en el JSON de la propia tabla). Pero para la creación de entidades con Big Query bastaría lo que sigue a continuación:

In [8]:
## Creación de entidades

customers = [
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("country", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("registration_date", "DATE", mode="REQUIRED"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.customers"
table = bigquery.Table(table_ref, schema=customers)
table = client.create_table(table, exists_ok=True)
print(f"Tabla customers creada")

categories = [
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.categories"
table = bigquery.Table(table_ref, schema=categories)
table = client.create_table(table, exists_ok=True)
print("Tabla categories creada")


products = [
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("cost", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("stock", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("is_active", "BOOLEAN", mode="NULLABLE"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.products"
table = bigquery.Table(table_ref, schema=products)
table = client.create_table(table, exists_ok=True)
print("Tabla products creada")



order_items = [
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("unit_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("discount_pct", "NUMERIC", mode="NULLABLE"),
]
schema  = order_items
table_ref = f"{PROJECT_ID}.{DATASET_ID}.order_items"
table = bigquery.Table(table_ref, schema=order_items)
table = client.create_table(table, exists_ok=True)
print("Tabla order_items creada")




orders = [
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("status", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("shipping_address", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("shipping_city", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("shipping_country", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("order_date", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("shipped_date", "TIMESTAMP", mode="NULLABLE"),
    bigquery.SchemaField("delivered_date", "TIMESTAMP", mode="NULLABLE"),
]
schema = orders
table_ref = f"{PROJECT_ID}.{DATASET_ID}.orders"
table = bigquery.Table(table_ref, schema=orders)
table = client.create_table(table, exists_ok=True)
print("Tabla orders creada")


payments = [
    bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("method", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("status", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("amount", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("currency", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("paid_at", "TIMESTAMP", mode="NULLABLE"),
]
table_ref = f"{PROJECT_ID}.{DATASET_ID}.payments"
table = bigquery.Table(table_ref, schema=payments)
table = client.create_table(table, exists_ok=True)
print("Tabla payments creada")



reviews = [
    bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]
table_ref = f"{PROJECT_ID}.{DATASET_ID}.reviews"
table = bigquery.Table(table_ref, schema=reviews)
table = client.create_table(table, exists_ok=True)
print("Tabla reviews creada")

Tabla customers creada
Tabla categories creada
Tabla products creada
Tabla order_items creada
Tabla orders creada
Tabla payments creada
Tabla reviews creada
